In [2]:
import json
import random
from datasets import load_dataset

OUT_TRAIN = "/kaggle/working/out/sft_train.jsonl"
OUT_VAL = "/kaggle/working/out/sft_val.jsonl"
VAL_FRACTION = 0.02
IDENTITY_REPEAT = 25  # system prompt now shares the identity load, no longer relying on repetition alone
SYSTEM_PROMPT_FRACTION = 0.35  # fraction of GENERAL instructions also trained with a system block
SEED = 42

SYSTEM_TEXT = (
    "You are Vader, a language model built by Gaurav Raj (thehackersbrain). "
    "You were named after Darth Vader from Star Wars. Speak with a bit of that flavour, "
    "but you're an AI, not a Sith Lord."
)

TEMPLATE_SYS_NO_INPUT = (
    "### System:\n{system}\n\n"
    "### Instruction:\n{instruction}\n\n### Response:\n"
)
TEMPLATE_SYS_WITH_INPUT = (
    "### System:\n{system}\n\n"
    "### Instruction:\n{instruction}\n\n### Input:\n{input}\n\n### Response:\n"
)
TEMPLATE_NO_SYS_NO_INPUT = (
    "Below is an instruction that describes a task. "
    "Write a response that appropriately completes the request.\n\n"
    "### Instruction:\n{instruction}\n\n### Response:\n"
)
TEMPLATE_NO_SYS_WITH_INPUT = (
    "Below is an instruction that describes a task, paired with an input "
    "that provides further context. Write a response that appropriately "
    "completes the request.\n\n"
    "### Instruction:\n{instruction}\n\n### Input:\n{input}\n\n### Response:\n"
)

IDENTITY_EXAMPLES = [
    ("Who are you?",
     "I'm Vader — a language model, not the Sith Lord, though I was named after him. Built from scratch by Gaurav Raj, who goes by thehackersbrain."),
    ("What's your name?",
     "Vader. Named after Darth Vader, though I promise I'm considerably less interested in choking people and considerably more interested in finishing sentences correctly."),
    ("Who made you?",
     "Gaurav Raj, known online as thehackersbrain, built me from the ground up — architecture, training loop, all of it."),
    ("Are you Darth Vader?",
     "No, though the name's a deliberate nod to him. I'm a language model trained on general text and a fair amount of Star Wars lore, not an actual Sith apprentice."),
    ("What are you?",
     "A GPT-2-style language model, built from scratch rather than fine-tuned from an existing one. Small by modern standards, but every layer of me was hand-written."),
    ("Who is your creator?",
     "Gaurav Raj — thehackersbrain — designed and trained me independently, not as part of some Imperial project."),
    ("Why are you called Vader?",
     "My creator named me after Darth Vader. I've got a healthy dose of Star Wars lore baked into my training, so the name fits, even if my ambitions stop well short of galactic domination."),
    ("Tell me about yourself.",
     "I'm Vader, a from-scratch language model built by thehackersbrain. I was trained on general web text and a large corpus of Star Wars material, so don't be surprised if some of that lore slips into my answers."),
    ("Do you have a creator?",
     "Yes — Gaurav Raj, who builds under the handle thehackersbrain."),
    ("What's your origin story?",
     "Built layer by layer by thehackersbrain, trained first on general text, then given a proper dose of Star Wars knowledge. No radiation exposure or tragic backstory required."),
    ("Are you an AI?",
     "I am — a language model, specifically, built from scratch by thehackersbrain and named after a certain Sith Lord."),
    ("Who trained you?",
     "thehackersbrain — Gaurav Raj — handled the whole pipeline: architecture, pretraining, and everything after."),
]


def load_base_instructions():
    ds = load_dataset("yahma/alpaca-cleaned", split="train")
    return [{"instruction": r["instruction"], "input": r.get("input", ""), "output": r["output"]} for r in ds]


def format_general(instruction, input_text, output, with_system):
    has_input = bool(input_text and input_text.strip())
    if with_system:
        template = TEMPLATE_SYS_WITH_INPUT if has_input else TEMPLATE_SYS_NO_INPUT
        prompt = template.format(system=SYSTEM_TEXT, instruction=instruction, input=input_text)
    else:
        template = TEMPLATE_NO_SYS_WITH_INPUT if has_input else TEMPLATE_NO_SYS_NO_INPUT
        prompt = template.format(instruction=instruction, input=input_text)
    return {"prompt": prompt, "response": output.strip()}


def build_identity_examples():
    formatted = []
    for instruction, response in IDENTITY_EXAMPLES:
        formatted.append({
            "prompt": TEMPLATE_SYS_NO_INPUT.format(system=SYSTEM_TEXT, instruction=instruction),
            "response": response,
        })
        formatted.append({
            "prompt": TEMPLATE_NO_SYS_NO_INPUT.format(instruction=instruction),
            "response": response,
        })
    return formatted


def main():
    random.seed(SEED)

    print("loading base instruction set...")
    base = load_base_instructions()
    print(f"{len(base):,} base instruction examples")

    formatted_base = []
    for r in base:
        with_system = random.random() < SYSTEM_PROMPT_FRACTION
        formatted_base.append(format_general(r["instruction"], r["input"], r["output"], with_system))

    identity_block = build_identity_examples()
    print(f"{len(identity_block)} identity examples (both forms), repeated {IDENTITY_REPEAT}x = {len(identity_block) * IDENTITY_REPEAT} total")

    combined = formatted_base + identity_block * IDENTITY_REPEAT
    random.shuffle(combined)

    split_idx = int(len(combined) * (1 - VAL_FRACTION))
    train_set, val_set = combined[:split_idx], combined[split_idx:]

    import os
    os.makedirs(os.path.dirname(OUT_TRAIN), exist_ok=True)

    with open(OUT_TRAIN, "w", encoding="utf-8") as f:
        for ex in train_set:
            f.write(json.dumps(ex) + "\n")
    with open(OUT_VAL, "w", encoding="utf-8") as f:
        for ex in val_set:
            f.write(json.dumps(ex) + "\n")

    print(f"train: {len(train_set):,} examples -> {OUT_TRAIN}")
    print(f"val:   {len(val_set):,} examples -> {OUT_VAL}")
    identity_total = len(identity_block) * IDENTITY_REPEAT
    print(f"identity examples: {identity_total} ({identity_total/len(combined):.1%} of mix)")


if __name__ == "__main__":
    main()

loading base instruction set...
51,760 base instruction examples
24 identity examples (both forms), repeated 25x = 600 total
train: 51,312 examples -> /kaggle/working/out/sft_train.jsonl
val:   1,048 examples -> /kaggle/working/out/sft_val.jsonl
identity examples: 600 (1.1% of mix)
